# Згадки про персонажів

## Видобування та таблиця

In [1]:
import json
from collections import Counter

import pandas as pd

In [2]:
INPUT_FILE = "../lab01/Zabrodin_file_2.json"
OUTPUT_FILE = "Zabrodin_characters.csv"


def extract_names(file_path):
    candidates = []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    result = data.get("result", "")

    for line in result.splitlines():
        if not line or line.startswith("#"):
            continue

        fields = line.split("\t")
        if len(fields) < 10:
            continue

        lemma = fields[2]
        pos_tag = fields[3]
        feats = fields[5]

        if pos_tag == "PROPN":
            if "NameType=Giv" in feats or "NameType=Sur" in feats:
                candidates.append(lemma.strip().title())

    return candidates


names_list = extract_names(INPUT_FILE)
character_counts = Counter(names_list)
top_characters = character_counts.most_common()

df = pd.DataFrame(top_characters, columns=["Персонаж", "Частота"])

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

display(df)

,Персонаж,Частота
0,Мотря,431
1,Лаврін,327
2,Кайдашиха,316
3,Карпо,280
4,Кайдаш,253
...,...,...
118,Меланка,1
119,Василь,1
120,Груша,1
121,Груш,1


Отримано список імен. Серед них є однакові імена, але у різних відмінках або імена з помилкою у написані, через що вони були розпізнані як різні. Також у списку є і не імена зовсім. Проте імена головних героїв та деяких другорядних були розпізнані.

## Дослідження

Для дослідження імен було обрано топ-10 без тих, що є помилковими.

In [3]:
characters = (
    "Мотря",
    "Лаврін",
    "Кайдашиха",
    "Карпо",
    "Кайдаш",
    "Мелашка",
    "Палажка",
    "Балаш",
    "Параска",
    "Довбиш"
)

VOWELS = set("аеєиіїоуюя")
SOFT_SIGN = set("ьʼ'")
SOFTENERS = set("ьіяює")
SONORANTS = set("вйлмнр")
VOICELESS = set("кптфхцчшщс")
VOICED = set("бгґджз")


def get_phonetic_profile(name):
    name_lower = name.lower()

    vowels = [c for c in name_lower if c in VOWELS]
    consonants = [c for c in name_lower if c.isalpha() and c not in VOWELS and c not in SOFT_SIGN]

    consonants_count = len(consonants)
    vowels_count = len(vowels)

    sonorants_c = sum(1 for c in consonants if c in SONORANTS)
    voiceless_c = sum(1 for c in consonants if c in VOICELESS)
    voiced_c = sum(1 for c in consonants if c in VOICED)

    soft_c = 0
    hard_c = 0

    for i, char in enumerate(name_lower):
        if char.isalpha() and char not in VOWELS and char not in SOFT_SIGN:
            if char == 'й':
                soft_c += 1
            elif i + 1 < len(name_lower) and name_lower[i + 1] in SOFTENERS:
                soft_c += 1
            else:
                hard_c += 1

    melody_index = round(sonorants_c / consonants_count, 2) if consonants_count else 0
    harshness_index = round(voiceless_c / consonants_count, 2) if consonants_count else 0
    vc_ratio = round(vowels_count / consonants_count, 2) if consonants_count else 0

    return {
        "Персонаж": name,
        "Довжина": len(name_lower),
        "Голосні (V)": vowels_count,
        "Приголосні (C)": consonants_count,
        "З них дзвінкі": voiced_c,
        "З них глухі": voiceless_c,
        "З них сонорні": sonorants_c,
        "З них тверді": hard_c,
        "З них м'які": soft_c,
        "V/C Відношення": vc_ratio,
        "Частка сонорних": melody_index,
        "Частка глухих": harshness_index,
    }


df_phonetics = pd.DataFrame([get_phonetic_profile(n) for n in characters])

df_phonetics.to_csv("Zabrodin_characters_analysis.csv", index=False, encoding="utf-8")
display(df_phonetics)

,Персонаж,Довжина,Голосні (V),Приголосні (C),З них дзвінкі,З них глухі,З них сонорні,З них тверді,З них м'які,V/C Відношення,Частка сонорних,Частка глухих
0,Мотря,5,2,3,0,1,2,2,1,0.67,0.67,0.33
1,Лаврін,6,2,4,0,0,4,3,1,0.50,1.00,0.00
2,Кайдашиха,9,4,5,1,3,1,4,1,0.80,0.20,0.60
3,Карпо,5,2,3,0,2,1,3,0,0.67,0.33,0.67
4,Кайдаш,6,2,4,1,2,1,3,1,0.50,0.25,0.50
5,Мелашка,7,3,4,0,2,2,4,0,0.75,0.50,0.50
6,Палажка,7,3,4,1,2,1,4,0,0.75,0.25,0.50
7,Балаш,5,2,3,1,1,1,3,0,0.67,0.33,0.33
8,Параска,7,3,4,0,3,1,4,0,0.75,0.25,0.75
9,Довбиш,6,2,4,2,1,1,4,0,0.50,0.25,0.25


* **V/C відношення**. У цій характеристиці прослідковується така закономірність: чотири імені з найбільшим значенням є іменами жінок у цьому творі. Натомість найменші значення мають чоловіки.
* **Частка сонорних**. У цій характеристиці можна побачити, що в імені Лаврін усі приголосні є сонорними. Тобто ім'я є мелодичним та м'яким, цю ж особливість можна побачити у характері цього персонажу. Крім цього двома іншими іменами з найбільшою часткою сонорних приголосних є Мотря та Мелашка: дві молоді головні героїні, яким не дивлячись на свою жіночність на початку твору, довелось стати на свій захист будь-то словесно чи вже фізично.
* **Частка глухих**. Тут проглядається така закономірність: два головні герої Кайдаш та Кайдашиха мають одні з найбільших значень, вони є старими, але сильними героями, у можливостях яких зазвичай не сумніваються, їх поважають. На противагу м'якості імені Лавріна, Карпо має більше значення частки глухих, це можна пов'язати з тим, що він є старшим братом.
* Окремим випадком можна прослідити майже однаковість характеристик імен Палажка та Параска. Вони є два другорядними героїнями твору, що вже у літах. Вони у багатому чому є схожими: пліткують, брешуть, намагаються вилікувати старого Кайдаша від пияцтва та обидві намагаються завоювати повагу до себе у селі.
* **Аптоніми**. Ім'я старого Кайдаша може бути аптонімом від слова кайдани, оскільки цього героя можна схарактеризувати як одне з джерел проблем у родині Кайдашів, через своє пияцтво він не здатен утримати мир у родині. Таким чином Кайдашиха, завдяки суфіксу '_-иха_' якоюсь мірою додає ще більше проблем у родину. Ім'я Мотря може мати зв'язок зі словом мотати, оскільки її характер мотає нерви всій родині, або молоти, бо вона 'молотить язиком'. Лаврін може бути пов'язаний зі словом лаври, він є людяним, стійким до інтриг, зберігає моральні цінності попри проблеми у родині.
* Суфікс _-ка_ в імені Мелашки надає йому відтінку зменшеності та пестливості, що відповідає її статусу наймолодшої невістки в родині